# توجيه الوكيل: مختبر عملي

في هذا المختبر ستتعلم كيفية بناء نماذج مطالبات (Prompts) فعالة لوكلاء الذكاء الاصطناعي، بما في ذلك استخدام أدوار النظام والمستخدم والمساعد، ونمط ReAct، والمطالبات الديناميكية. ستستخدم بايثون لإنشاء قوالب وإجراء اختبارات عملية.

المتطلبات: المكتبات المدعومة محملة مسبقاً، ومفتاح DeepSeek API متوفر.

In [ ]:
import os
import re
import numpy as np

# تعيين البذرة للحصول على نتائج قابلة للتكرار
np.random.seed(42)

# مفتاح API من المتغيرات البيئية
api_key = os.environ.get("DEEPSEEK_API_KEY")
if api_key:
    print("API key loaded successfully.")
else:
    print("Warning: API key not found. بعض الأجزاء قد لا تعمل.")

## أنواع المطالبات (Prompt Types)

في وكلاء الذكاء الاصطناعي، تتكون المحادثات عادةً من ثلاث أدوار للرسائل:
- **النظام (System)**: يحدد سياق وسلوك الوكيل (مثال: "أنت مساعد مفيد تتحدث العربية").
- **المستخدم (User)**: استفسارات أو أوامر من الشخص.
- **المساعد (Assistant)**: ردود الوكيل النموذج.

في الخلية التالية، سيطلب منك بناء قائمة رسائل تحتوي على هذه الأدوار.

In [ ]:
# بناء قائمة رسائل لمحادثة مع وكيل مساعد
messages = [
    {"role": "system", "content": "# TODO: اكتب محتوى رسالة النظام لتعريف الوكيل بأنه مساعد مفيد يتحدث العربية"},
    {"role": "user", "content": "ما هي عاصمة مصر؟"},
]

# عرض المحادثة الأولية
print("System message:", messages[0]["content"])
print("User message:", messages[1]["content"])
print("(سيتم إضافة رد المساعد لاحقاً)")

## نمط ReAct (Reason + Act)

نمط ReAct هو أسلوب للتوجيه حيث يقوم الوكيل بالتفكير (Thought) قبل تنفيذ إجراء (Action)، ثم يراقب النتيجة (Observation). يتم تكرار هذه الدورة.

الهيكل النصي:
```
Thought: [تفكير]
Action: [اسم الأداة]
Action Input: [مدخلات]
Observation: [نتيجة]
...
Thought: [تفكير نهائي]
Final Answer: [الإجابة]
```

في التطبيق التالي، ستقوم بإنشاء نص موجه بأسلوب ReAct.

In [ ]:
def build_react_prompt(user_query, tools, examples):
    """
    يبني نص المطالبة بنمط ReAct.
    """
    tools_desc = ""
    for tool_name, tool_desc in tools.items():
        tools_desc += f"- {tool_name}: {tool_desc}\n"
    
    examples_str = ""
    for ex in examples:
        examples_str += f"Question: {ex['question']}\n{ex['trajectory']}\n\n"
    
    prompt = f"""You are an assistant that uses the ReAct framework. You have access to the following tools:

{tools_desc}
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{', '.join(tools.keys())}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Examples:
{examples_str}
Question: {user_query}
Thought:"""
    return prompt

# TODO: قم بتعريف الأدوات (tools) على شكل قاموس يحتوي على اسم الأداة ووصفها.
# يجب أن تشمل أداة "Calculator" تقوم بعمليات حسابية بسيطة.
# وأكمل الأمثلة لتوضيح كيفية استخدام الأداة.

tools = {
    # TODO: أضف أداة Calculator مع وصفها
}

examples = [
    {
        "question": "What is 15 + 27?",
        "trajectory": "Thought: I need to calculate 15 + 27\nAction: Calculator\nAction Input: 15 + 27\nObservation: 42\nThought: I now know the final answer\nFinal Answer: 42"
    },
    # TODO: أضف مثالاً آخر يوضح عملية حسابية أخرى (مثال: طرح أو ضرب)
]

# اختبار الدالة
query = "What is 33 * 7?"
prompt = build_react_prompt(query, tools, examples)
print(prompt)

In [ ]:
def execute_calculator(action_input):
    """
    تقوم بتنفيذ العملية الحسابية من النص المدخل.
    تدعم الجمع (+) والطرح (-) والضرب (*) والقسمة (/).
    """
    # TODO: قم بتحليل النص action_input واستخراج العملية الرياضية،
    # مثلاً "15 + 27" يجب أن تفصل إلى أرقام وعامل.
    # ثم قم بحساب النتيجة وإعادتها.
    # استخدم التعبيرات النمطية (regular expressions) أو split.
    pass

# اختبار بسيط
print(execute_calculator("15 + 27"))  # المفترض 42
print(execute_calculator("33 * 7"))  # المفترض 231

## المطالبات الديناميكية (Dynamic Prompts)

غالباً ما تحتاج المطالبات إلى تضمين متغيرات مثل اسم المستخدم أو قائمة الأدوات المتاحة. لتحقيق ذلك، نستخدم قوالب نصوص (string templates) في بايثون.

في التمرين التالي، ستقوم بإنشاء قالب لمطالبة وكيل شخصي، ثم ملؤه بمتغيرات.

In [ ]:
# قالب لمطالبة وكيل شخصي
template = """
أنت وكيل شخصي اسمه {agent_name}. مهمتك مساعدة {user_name}.
الأدوات المتاحة لديك:
{tools_list}
استجب دائماً باللغة العربية وبأسلوب ودود.
"""

# TODO: استخدم الدالة format() لملء المتغيرات:
# agent_name = "مساعد", user_name = "أحمد", tools_list = "- البحث\n- الحاسبة"
# ثم اطبع النتيجة.

# أضف السطر المناسب هنا:

In [ ]:
# تجربة القالب مع أسماء مختلفة
names = ["سارة", "خالد", "منى"]
for name in names:
    prompt = template.format(agent_name="مساعد", user_name=name, tools_list="- البحث\n- الحاسبة")
    print(f"--- Prompt for {name} ---")
    print(prompt)
    print("\n")

## تسلسل الأفكار (Chain-of-Thought)

تقنية CoT تطلب من النموذج التفكير خطوة بخطوة قبل الإجابة، مما يحسن الدقة في المسائل المنطقية.

في الخلية التالية، ستكتب مطالبة تحث النموذج على حل مسألة حسابية عبر التفكير المتسلسل.

In [ ]:
def build_cot_prompt(problem):
    """
    تنشئ مطالبة CoT لمشكلة رياضية.
    """
    # TODO: اكتب النص الذي يطلب من النموذج التفكير خطوة بخطوة،
    # ثم تقديم الإجابة النهائية.
    cot_prompt = f"""..."""  # قم بتعبئة الهيكل
    return cot_prompt

# اختبار
problem = "إذا كان لدى شخص 10 تفاحات، وأعطى 3 لأخيه، ثم اشترى 5، كم تفاحة لديه؟"
print(build_cot_prompt(problem))

In [ ]:
if api_key:
    # TODO: استخدم مكتبة requests لاستدعاء DeepSeek API
    # أرسل المطالبة التي بنيتها (prompt من خلية سابقة) إلى النموذج.
    # استخدم النموذج "deepseek-chat" واطبع الرد.
    pass
else:
    print("لا يمكن استدعاء API بدون مفتاح.")

## خلاصة

في هذا المختبر، تعلمت:
- بناء قوائم رسائل بأدوار النظام والمستخدم والمساعد.
- إنشاء مطالبات بنمط ReAct تشمل وصف الأدوات والأمثلة.
- تحليل مخرجات الأداة من الإجراءات.
- استخدام القوالب الديناميكية لملء المتغيرات.
- تطبيق تقنية تسلسل الأفكار (CoT).
- استدعاء وكلاء حقيقيين باستخدام DeepSeek API.

هذه المهارات أساسية لتوجيه وكلاء الذكاء الاصطناعي بفعالية.